# Watermark U-Net training (Kaggle GPU)

1. Settings → Accelerator → **GPU** (T4/P100, quota ~30h/week).
2. Clone once into /kaggle/working, then `%cd` into it (avoid nested clones):
   `!git clone https://github.com/AliTabibAzar/hamrah-watermark.git /kaggle/working/hamrah-watermark`
3. Run cells top to bottom. Weights land in `models/` — download `watermark-unet.pt` when done.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
%cd /kaggle/working/hamrah-watermark
!pip install -q -r requirements-train.txt

In [ ]:
# 1) real datasets (needs internet, CPU-only cell is fine)
!python scripts/download_data.py --out data --sets pita
# 2) unzip/copy CLWD + LOGO-* into data/clwd, data/logo-l, ... (images/ + masks/ layout)
# 3) flatten PITA snapshot into plain backgrounds, then synthetic top-up (fast, CPU)
!python scripts/convert_pita.py --src data/pita_raw --dst data/pita_bg
!python scripts/gen_synthetic.py --bg data/pita_bg --out data/synth --n 5000

In [ ]:
# Train. ~40 epochs, AMP on, early stopping. A few hours on a T4.
!python train_unet.py --data data/clwd data/logo-l data/logo-h data/synth --out models --epochs 40 --batch 16 --size 256

In [ ]:
# Sanity check: IoU on a held-out batch + file size
!ls -la models/watermark-unet.pt
!python - <<'EOF'
import torch, glob
from src.data.watermark_data import MaskDataset
from src.models.unet import WatermarkUNet
from train_unet import iou_score
ds = MaskDataset(sorted(glob.glob('data/*/'))[0])
m = WatermarkUNet(pretrained_encoder=False)
m.load_state_dict(torch.load('models/watermark-unet.pt', map_location='cpu'))
m.eval()
ious = []
for i in range(min(50, len(ds))):
    x, y = ds[i]
    with torch.no_grad():
        ious.append(iou_score(m(x[None]), y[None]))
print('mean IoU on 50 samples:', round(sum(ious) / len(ious), 3))
EOF